# Get embeddings

In [1]:
%load_ext autoreload

%autoreload 2

In [1]:
import numpy as np
import pandas as pd
import pathlib
import torch
from final_project_package.ml_logic.data_clean import initialize_clip, data_clean, add_clip_columns, average_scoring
from final_project_package.ml_logic.model import initialize_model, train_model, evaluate_model
from final_project_package.ml_logic.preprocessor_pipeline import get_fitted_preprocessor
from final_project_package.embeddings.embeddings import load_clip_model, get_image_embeddings
from final_project_package.embeddings.embeddings_frontend import get_text_embeddings


✅ TensorFlow loaded (0.22s)


In [4]:
path_to_project = "."
data_path = pathlib.Path(path_to_project)

# import images.csv
image_full = pd.read_csv(data_path / "data_dump/images_cleaned.csv")

# import listings.csv
listings_full = pd.read_csv(data_path / "data_dump/listings_cleaned.csv")
listings_full = listings_full[10:20]
source_id = listings_full["source_id"]
image_full = image_full[image_full["source_id"].isin(source_id)]

image_full["image_path"] = image_full["image_name"].apply(lambda x: \
    str(data_path / "raw_data/suumo_images" / str(x).split("_")[0] / x))

# initialize clip
model, processor = load_clip_model()
print("clip initialized")

image_df = image_full

Loading CLIP model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


clip initialized


In [5]:
len(image_df)

103

In [6]:
# add embedding column
images = image_df["image_path"]
images[0:2].apply(lambda x: \
    get_image_embeddings(model, processor, [x]))


87    [-0.07538603246212006, -0.004289701581001282, ...
88    [-0.054156746715307236, -0.02384055405855179, ...
Name: image_path, dtype: object

In [7]:
image_emb = pd.read_csv(data_path / "data_dump/images_cleaned_embedding.csv")
image_emb[100:103]

,source_id,listing_url,image_url,image_name,image_path,default_image,room_type,scoring_dict,embedding
100,20139608,https://suumo.jp/ms/chuko/tokyo/sc_nerima/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20139608_af026c0e11.jpg,/home/bea/code/katiarojas87/final-project/raw_...,1,kitchen,"{'luxury': 0.40625, 'brightness': 0.73046875, ...","[-0.0447964146733284, -0.020617032423615456, 0..."
101,20139608,https://suumo.jp/ms/chuko/tokyo/sc_nerima/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20139608_a93f632003.jpg,/home/bea/code/katiarojas87/final-project/raw_...,1,kitchen,"{'luxury': 0.59375, 'brightness': 0.46875, 'co...","[-0.030182791873812675, 0.023825937882065773, ..."
102,20139608,https://suumo.jp/ms/chuko/tokyo/sc_nerima/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20139608_c9f0ac1a1a.jpg,/home/bea/code/katiarojas87/final-project/raw_...,1,bedroom,"{'luxury': 0.62109375, 'brightness': 0.7304687...","[-0.031536929309368134, 0.01687052473425865, -..."


In [8]:
text_embedding = get_text_embeddings(model, processor, ["skandenavian kitchen"])
text_embedding.dim()

Generating text embeddings for labels...


1

In [9]:
get_image_embeddings(model, processor, image_emb["image_path"][100])

[-0.0447964146733284,
 -0.020617032423615456,
 0.015363971702754498,
 -0.011093738488852978,
 0.029478812590241432,
 -0.010711017064750195,
 0.0021741001401096582,
 0.03156687319278717,
 0.06648311018943787,
 -0.013273024931550026,
 0.0245360154658556,
 -0.020485127344727516,
 0.033868078142404556,
 -0.08101608604192734,
 0.028119539842009544,
 -0.04601723328232765,
 -0.030788270756602287,
 0.007843714207410812,
 0.0044043054804205894,
 -0.015293462201952934,
 0.05349224805831909,
 0.026050927117466927,
 0.02790329046547413,
 0.08097783476114273,
 0.008276643231511116,
 0.0402560718357563,
 0.04255983605980873,
 -0.043750010430812836,
 0.011484350077807903,
 -0.01144667249172926,
 0.013095105066895485,
 0.017923275008797646,
 -0.038673583418130875,
 -0.025976093485951424,
 0.03159451484680176,
 0.01645306870341301,
 0.009650589898228645,
 -0.006222404073923826,
 0.01940527930855751,
 0.2207464873790741,
 -0.026987411081790924,
 -0.006060739513486624,
 0.030138589441776276,
 -0.03701033

In [10]:
torch.tensor(eval(image_emb["embedding"][100])).dim()

1

In [11]:
%time
torch.nn.CosineSimilarity(dim=0)(text_embedding, torch.tensor(eval(image_emb["embedding"][100])))

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.53 μs


tensor(0.2556)

In [17]:
%time
similarity = (torch.tensor(eval(image_emb["embedding"][100])) @ torch.tensor(text_embedding).T).numpy()
similarity, similarity == 0.2556486, similarity > 0.6

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.29 μs


/tmp/ipykernel_514545/4141203439.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  similarity = (torch.tensor(eval(image_emb["embedding"][100])) @ torch.tensor(text_embedding).T).numpy()


(array(0.2556486, dtype=float32), np.True_, np.False_)